# Iterative-Linear State Estimation Uncertainty Quantification

This notebook shows how to calculate analytical output standard deviations for iterative-linear (IL) state estimation. It covers:

- constructing a small observable network and its measurements;
- enabling uncertainty quantification with `calculate_uncertainty=True`;
- reading voltage, current, and active/reactive-power sigma fields; and
- checking how output uncertainty scales with measurement uncertainty.

The reported values are first-order standard deviations for the final, fixed local IL model. See [State Estimate Output Uncertainty](../algorithms/se-algorithms.md#state-estimate-output-uncertainty) for the mathematical assumptions.

In [1]:
import power_grid_model
from power_grid_model._core.power_grid_model_c.get_pgm_dll_path import (
    get_pgm_dll_path,
)

print("Python package:", power_grid_model.__file__)
print("Native library:", get_pgm_dll_path())

Python package: /home/jhe/pgm_dev/power-grid-model/src/power_grid_model/__init__.py
Native library: /home/jhe/pgm_dev/power-grid-model/build/bin/libpower_grid_model_c.so


In [2]:
import sys

import pandas as pd

import power_grid_model

print(sys.executable)
print(pd.__version__)
print(power_grid_model.__file__)

/home/jhe/pgm_dev/power-grid-model/.venv/bin/python
3.0.5
/home/jhe/pgm_dev/power-grid-model/src/power_grid_model/__init__.py


In [3]:
import numpy as np
import pandas as pd

from power_grid_model import (
    AttributeType,
    CalculationMethod,
    CalculationType,
    ComponentType,
    DatasetType,
    LoadGenType,
    MeasuredTerminalType,
    PowerGridModel,
    initialize_array,
)
from power_grid_model.validation import assert_valid_input_data

## Network and measurements

The example uses a two-node network. A voltage-magnitude sensor observes the source node, a power-flow sensor observes the line, and a load power sensor provides the remaining measurement information.

```text
source 41 -- node 11 -- line 21 -- node 12 -- load 31
              |                         |
       voltage sensor 71          power sensor 62
                        line power sensor 61
```

All values use SI units. Sensor sigma attributes describe input measurement errors; they are different from the output sigma attributes calculated below.

In [4]:
node = initialize_array(DatasetType.input, ComponentType.node, 2)
node[AttributeType.id] = [11, 12]
node[AttributeType.u_rated] = [10.5e3, 10.5e3]

line = initialize_array(DatasetType.input, ComponentType.line, 1)
line[AttributeType.id] = [21]
line[AttributeType.from_node] = [11]
line[AttributeType.to_node] = [12]
line[AttributeType.from_status] = [1]
line[AttributeType.to_status] = [1]
line[AttributeType.r1] = [0.1]
line[AttributeType.x1] = [0.0]
line[AttributeType.c1] = [0.0]
line[AttributeType.tan1] = [0.0]
line[AttributeType.i_n] = [510.0]

source = initialize_array(DatasetType.input, ComponentType.source, 1)
source[AttributeType.id] = [41]
source[AttributeType.node] = [11]
source[AttributeType.status] = [1]
source[AttributeType.u_ref] = [1.0]

sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, 1)
sym_load[AttributeType.id] = [31]
sym_load[AttributeType.node] = [12]
sym_load[AttributeType.status] = [1]
sym_load[AttributeType.type] = [LoadGenType.const_power]

sym_voltage_sensor = initialize_array(DatasetType.input, ComponentType.sym_voltage_sensor, 1)
sym_voltage_sensor[AttributeType.id] = [71]
sym_voltage_sensor[AttributeType.measured_object] = [11]
sym_voltage_sensor[AttributeType.u_measured] = [10.5e3]
sym_voltage_sensor[AttributeType.u_sigma] = [105.0]

sym_power_sensor = initialize_array(DatasetType.input, ComponentType.sym_power_sensor, 2)
sym_power_sensor[AttributeType.id] = [61, 62]
sym_power_sensor[AttributeType.measured_object] = [21, 31]
sym_power_sensor[AttributeType.measured_terminal_type] = [
    MeasuredTerminalType.branch_from,
    MeasuredTerminalType.load,
]
sym_power_sensor[AttributeType.p_measured] = [1.0e6, 2.0e6]
sym_power_sensor[AttributeType.q_measured] = [0.0, 0.0]
sym_power_sensor[AttributeType.power_sigma] = [1.0e3, 1.0e3]

input_data = {
    ComponentType.node: node,
    ComponentType.line: line,
    ComponentType.source: source,
    ComponentType.sym_load: sym_load,
    ComponentType.sym_voltage_sensor: sym_voltage_sensor,
    ComponentType.sym_power_sensor: sym_power_sensor,
}

assert_valid_input_data(input_data, calculation_type=CalculationType.state_estimation)

## Run IL state estimation with UQ

Sigma fields are part of the output schema for every calculation, but remain `NaN` unless UQ is explicitly requested. UQ is currently supported only by the iterative-linear state-estimation method.

In [5]:
model = PowerGridModel(input_data, system_frequency=50.0)

result_without_uq = model.calculate_state_estimation(
    calculation_method=CalculationMethod.iterative_linear,
)
node_u_sigma_without_uq = result_without_uq[ComponentType.node][AttributeType.u_sigma]
np.testing.assert_equal(np.count_nonzero(np.isnan(node_u_sigma_without_uq)), node_u_sigma_without_uq.size)

result = model.calculate_state_estimation(
    calculation_method=CalculationMethod.iterative_linear,
    calculate_uncertainty=True,
)

## Inspect node uncertainty

Each sigma has the same physical unit and shape as its corresponding output. Angle sigmas are in radians. Because this example has no angle measurement, phase A of the slack bus is the angle reference and its `u_angle_sigma` is zero.

In [6]:
node_columns = [
    "id",
    "u_pu",
    "u_pu_sigma",
    "u",
    "u_sigma",
    "u_angle",
    "u_angle_sigma",
    "p",
    "p_sigma",
    "q",
    "q_sigma",
]
node_result = pd.DataFrame(result[ComponentType.node])[node_columns]
np.testing.assert_equal(node_result.loc[0, "u_angle_sigma"], 0.0)
node_result

,id,u_pu,u_pu_sigma,u,u_sigma,u_angle,u_angle_sigma,p,p_sigma,q,q_sigma
0,11,1.000000,0.007071,10499.999990,74.246212,0.0,0.00000,1.501364e+06,10628.011975,0.0,10628.011975
1,12,0.998638,0.007071,10485.701289,74.246212,0.0,0.00001,-1.499319e+06,10627.947997,0.0,10628.011975


## Inspect branch uncertainty

For a two-terminal branch, current-magnitude and active/reactive-power sigmas are available at both sides. There is no `s_sigma`; active and reactive uncertainty are reported separately.

In [7]:
line_columns = [
    "id",
    "p_from",
    "p_from_sigma",
    "q_from",
    "q_from_sigma",
    "i_from",
    "i_from_sigma",
    "p_to",
    "p_to_sigma",
    "q_to",
    "q_to_sigma",
    "i_to",
    "i_to_sigma",
]
line_result = pd.DataFrame(result[ComponentType.line])[line_columns]
line_result

,id,p_from,p_from_sigma,q_from,q_from_sigma,i_from,i_from_sigma,p_to,p_to_sigma,q_to,q_to_sigma,i_to,i_to_sigma
0,21,1.501364e+06,10628.011975,0.0,10628.011975,82.553591,0.027493,-1.499319e+06,10627.947997,-0.0,10628.011975,82.553591,0.027493


## Compare input and output uncertainty

The voltage sensor directly supplies a magnitude sigma. The power sensors instead supply an apparent-power sigma: under PGM's circular model, each sensor contributes $\sigma_P=\sigma_Q=\sigma_S/\sqrt{2}$. In this simple series network, the two independent power sensors constrain the same current, giving another factor of $1/\sqrt{2}$ when their information is combined.

The output power sigma is not just the combined input power-component sigma. Reconstructing $S=UI^*$ also propagates the absolute-voltage uncertainty. The table below makes those contributions explicit. These simplified checks are specific to this two-node example.

In [8]:
input_voltage_sigma = float(sym_voltage_sensor[AttributeType.u_sigma][0])
input_apparent_power_sigma = float(sym_power_sensor[AttributeType.power_sigma][0])
input_component_power_sigma = input_apparent_power_sigma / np.sqrt(2.0)
combined_component_power_sigma = input_component_power_sigma / np.sqrt(sym_power_sensor.size)

measured_voltage = float(sym_voltage_sensor[AttributeType.u_measured][0])
expected_current_sigma = combined_component_power_sigma / (np.sqrt(3.0) * measured_voltage)
reported_current_sigma = float(line_result.loc[0, "i_from_sigma"])
np.testing.assert_allclose(reported_current_sigma, expected_current_sigma)

p_from = abs(float(line_result.loc[0, "p_from"]))
u_from = float(node_result.loc[0, "u"])
i_from = float(line_result.loc[0, "i_from"])
voltage_power_contribution = p_from * float(node_result.loc[0, "u_sigma"]) / u_from
current_power_contribution = p_from * reported_current_sigma / i_from
expected_power_sigma = np.hypot(voltage_power_contribution, current_power_contribution)

np.testing.assert_allclose(line_result.loc[0, "p_from_sigma"], expected_power_sigma)
np.testing.assert_allclose(line_result.loc[0, "q_from_sigma"], expected_power_sigma)

pd.DataFrame(
    {
        "quantity": ["voltage magnitude", "current magnitude", "active power", "reactive power"],
        "unit": ["V", "A", "W", "var"],
        "input-derived sigma": [
            input_voltage_sigma,
            expected_current_sigma,
            combined_component_power_sigma,
            combined_component_power_sigma,
        ],
        "voltage contribution": [np.nan, np.nan, voltage_power_contribution, voltage_power_contribution],
        "predicted output sigma": [
            input_voltage_sigma / np.sqrt(2.0),
            expected_current_sigma,
            expected_power_sigma,
            expected_power_sigma,
        ],
        "reported output sigma": [
            node_result.loc[0, "u_sigma"],
            reported_current_sigma,
            line_result.loc[0, "p_from_sigma"],
            line_result.loc[0, "q_from_sigma"],
        ],
    }
)

            quantity unit  input-derived sigma  voltage contribution  \
0  voltage magnitude    V           105.000000                   NaN   
1  current magnitude    A             0.027493                   NaN   
2       active power    W           500.000000          10616.244089   
3     reactive power  var           500.000000          10616.244089   

   predicted output sigma  reported output sigma  
0               74.246212              74.246212  
1                0.027493               0.027493  
2            10628.011975           10628.011975  
3            10628.011975           10628.011975  

## Measurement-error scaling

For this fixed linear model, multiplying every measurement sigma by two multiplies every propagated output sigma by two. The point estimate remains unchanged because all relative measurement weights stay the same.

In [9]:
scaled_input_data = {component: values.copy() for component, values in input_data.items()}
scaled_input_data[ComponentType.sym_voltage_sensor][AttributeType.u_sigma] *= 2.0
scaled_input_data[ComponentType.sym_power_sensor][AttributeType.power_sigma] *= 2.0

scaled_result = PowerGridModel(scaled_input_data, system_frequency=50.0).calculate_state_estimation(
    calculation_method=CalculationMethod.iterative_linear,
    calculate_uncertainty=True,
)

sigma_fields = {
    ComponentType.node: ["u_pu_sigma", "u_sigma", "u_angle_sigma", "p_sigma", "q_sigma"],
    ComponentType.line: [
        "p_from_sigma",
        "q_from_sigma",
        "i_from_sigma",
        "p_to_sigma",
        "q_to_sigma",
        "i_to_sigma",
    ],
}
for component, fields in sigma_fields.items():
    for field in fields:
        np.testing.assert_allclose(scaled_result[component][field], 2.0 * result[component][field])

pd.DataFrame(
    {
        "node_id": result[ComponentType.node]["id"],
        "u_sigma [V]": result[ComponentType.node]["u_sigma"],
        "u_sigma after 2x input sigma [V]": scaled_result[ComponentType.node]["u_sigma"],
    }
)

,node_id,u_sigma [V],u_sigma after 2x input sigma [V]
0,11,74.246212,148.492424
1,12,74.246212,148.492424


## Interpretation and limitations

The output sigmas are local analytical approximations, not confidence bounds for the complete iterative process. The propagation adopts the proper (circular) effective complex-error model documented for PGM IL state estimation. Current-magnitude sigma is `NaN` at exactly zero current. Ideal-link flow sigmas and the individual injection sigmas of nodes joined by connected ideal links are also `NaN`. A case requiring numerical LU pivot perturbation raises `SparseMatrixError` during UQ.